# 14 · Unity Catalog registry and batch inference

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** databricks. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Register a signed model version; use an alias; score records and persist predictions.


## Step 1 — Train and log a model

This notebook is self-contained. CREATE MODEL and catalog/schema privileges are required. The register switch is off until you choose to create a model version.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
user=spark.sql('SELECT current_user()').first()[0]
mlflow.set_tracking_uri('databricks')
mlflow.set_experiment(f'/Users/{user}/ai_engineering_hands_on')

import hashlib
from pyspark.sql import functions as F
user = spark.sql('SELECT current_user()').first()[0]
catalog = spark.sql('SELECT current_catalog()').first()[0]
schema = 'ai_lab_' + hashlib.sha256(user.encode()).hexdigest()[:10]
# Uses the current catalog; change it to a catalog where you can CREATE SCHEMA.
namespace = f'`{catalog}`.`{schema}`'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert name.replace('_', '').isalnum()
    return f'{namespace}.`{name}`'
print('Course namespace:', namespace)

model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500)).fit(X_train,y_train)
with mlflow.start_run(run_name='registry_candidate'):
    info=mlflow.sklearn.log_model(model,artifact_path='model',input_example=X_train[:2],signature=infer_signature(X_train,model.predict(X_train)))
model_uri=info.model_uri
print(model_uri)


**Check your result:** A fresh logged artifact exists even with registry registration disabled.


## Step 2 — Register explicitly

A three-part Unity Catalog model name identifies the governed asset. Each rerun with REGISTER=True creates another version.


In [ ]:
REGISTER=False
model_name=f'{catalog}.{schema}.churn_demo'
if REGISTER:
    mlflow.set_registry_uri('databricks-uc')
    version=mlflow.register_model(model_uri,model_name)
    registry=mlflow.MlflowClient()
    registry.set_registered_model_alias(model_name,'Candidate',version.version)
    inference_uri=f'models:/{model_name}@Candidate'
else:
    inference_uri=model_uri
print('Inference source:',inference_uri)


**Check your result:** Disabled mode uses the logged run artifact; enabled mode creates a registered Candidate version.


## Step 3 — Score distributed records

Spark UDF model inference avoids collecting a large dataset. The model expects a 10-column NumPy tensor.


In [ ]:
batch=spark.createDataFrame([tuple([int(i)]+[float(v) for v in row]) for i,row in enumerate(X_test[:30])],
    'id long, '+', '.join(f'f{i} double' for i in range(10)))
predict=mlflow.pyfunc.spark_udf(spark,model_uri=inference_uri,result_type='long',env_manager='local')
scored=batch.withColumn('prediction',predict(F.array(*[F.col(f'f{i}') for i in range(10)])))
scored=scored.withColumn('model_uri',F.lit(inference_uri)).withColumn('scored_at',F.current_timestamp())
scored.write.format('delta').mode('overwrite').saveAsTable(table('batch_predictions'))
assert spark.table(table('batch_predictions')).count()==30
display(spark.table(table('batch_predictions')).limit(5))


**Check your result:** Thirty predictions with model lineage are saved. If the compute restricts model UDF environments, use lab 27 for bounded CPU scoring.


## Practice

Pin the registered model version for a reproducible batch instead of using a movable alias.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Resolve the alias using MlflowClient.get_model_version_by_alias, then score `models:/catalog.schema.model/<version>`. Store that resolved version with output.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
